# Miniprojekt AI og Data

**Sebastian Flaaris og Nikolaj Mysling**

*May 7, 2026*

---

# Notebook: Manglende data, imputering, relationelle databaser og tidsseriedata

**Miniprojekt – AI og Data**  
Denne notebook samler arbejdet fra workshop 1 og workshop 2 i ét miniprojekt. Første del undersøger manglende værdier i salgsdata, sammenligner flere imputeringsmetoder og strukturerer data relationelt i SQLite. Anden del undersøger kortsigtede udsving / støjlignende variation i klimadata og anvender vinduesmetoden som præprocessering.

## Opgavespørgsmål der besvares

### Workshop 1 – Manglende data og relationelle databaser
1. Identificér typer af datamangel.
2. Implementér/anvend minimum to dataimputeringsmetoder.
3. Sammenlign metoderne, bl.a. ved brug af en classifier med/uden imputering.
4. Strukturér data i minimum to relationelle tabeller/databaser.

### Workshop 2 – Tidsseriedata og vinduesmetoden
1. Vurder graden af kortsigtede udsving / støjlignende variation gennem datavisualisering.
2. Anvend passende præprocessering med vinduesmetoden.
3. Plot data før og efter præprocessering.

## Datasæt

- `Sales_with_NaNs_v1.3.csv`: salgsdata med manglende værdier.
- `Sales_without_NaNs_v1.3.csv`: referenceversion uden NaNs, kun brugt til evaluering af imputerede værdier.
- `DailyDelhiClimateTrain.csv` og `DailyDelhiClimateTest.csv`: daglige klimadata som tidsserie.

**Vigtig begrænsning:** I en realistisk opgave har man normalt ikke adgang til en komplet referencefil uden NaNs. Her bruges referencefilen udelukkende til at evaluere, hvor tæt imputeringerne kommer på de oprindelige værdier. Selve metodevalget vurderes stadig ud fra datasættet med manglende værdier.


# 0. Faglig ramme og metodeoverblik

## Manglende data
Dataimputering kan defineres som en præprocesseringsproces, hvor manglende eller ufuldstændige dataværdier bliver erstattet med nye værdier. Et centralt princip er, at de manglende værdier ikke blot er "tomme pladser"; de imputerede værdier bør være realistiske i forhold til datasættets rammer, fordelinger og sammenhænge.

I denne notebook undersøges missingness både visuelt og kvantitativt. Vi bruger:

- antal og procent manglende værdier pr. kolonne,
- korrelation mellem missingness-indikatorer,
- en simpel “missingness classifier”, der tester om manglende værdier kan forudsiges ud fra andre observerede variable.

## Imputering
Der anvendes tre metoder, så kravet om minimum to metoder opfyldes med en ekstra sammenligning:

1. **Median/mode-imputering**: numeriske værdier erstattes med median og kategoriske med hyppigste kategori.
2. **Random sampling-imputering**: manglende værdier erstattes ved at sample fra observerede værdier i samme kolonne.
3. **Regressions-/iterativ imputering**: numeriske værdier estimeres ud fra relationer til de øvrige numeriske variable, mens kategoriske værdier håndteres med mode.

Metoderne evalueres på to måder:

- **Imputeringskvalitet**: hvor tæt de imputerede værdier er på referencefilen uden NaNs.
- **Modelnytte**: en classifier trænes til at forudsige `Purchase_Made` med og uden imputering.



In [ ]:
# 1. Biblioteker og generelle indstillinger
from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [ ]:
base_candidates = [
    Path.cwd(),
    Path.home() / "Documents" / "AI-og-Data-Miniprojekt1",
    Path("/mnt/data"),
]

def find_file(filename):
    for base in base_candidates:
        direct_candidate = base / filename
        if direct_candidate.exists():
            return direct_candidate
        for candidate in base.rglob(filename):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Kunne ikke finde {filename}. Læg filen i samme mappe som notebooken eller workspace-mappen.")

sales_nan_path = find_file("Sales_with_NaNs_v1.3.csv")
sales_full_path = find_file("Sales_without_NaNs_v1.3.csv")
climate_train_path = find_file("DailyDelhiClimateTrain.csv")
climate_test_path = find_file("DailyDelhiClimateTest.csv")

print("Filer fundet:")
for p in [sales_nan_path, sales_full_path, climate_train_path, climate_test_path]:
    print("-", p)

# Del 1 – Manglende data, imputering og relationelle databaser

## 1.1 Indlæsning og første datatjek

**Overvejelse:** Før metodevalg undersøges datasættets størrelse, kolonner, datatyper og synlige mangler. Dette er nødvendigt, fordi numeriske og kategoriske variable kræver forskellige imputeringsstrategier.

In [ ]:
# Indlæs salgsdata
sales_nan = pd.read_csv(sales_nan_path)
sales_full = pd.read_csv(sales_full_path)

print("Sales med NaNs:", sales_nan.shape)
print("Sales uden NaNs:", sales_full.shape)

display(sales_nan.head())

overview = pd.DataFrame({
    "dtype": sales_nan.dtypes.astype(str),
    "non_null": sales_nan.notna().sum(),
    "missing": sales_nan.isna().sum(),
    "missing_pct": (sales_nan.isna().mean() * 100).round(2),
    "unique_values": sales_nan.nunique(dropna=True)
})
display(overview)

## 1.2 Visualisering af datamangel

**Overvejelse:** Hvis manglende værdier ligger tilfældigt spredt, peger det i retning af MCAR. Hvis de optræder i mønstre eller kan forudsiges ud fra andre variable, peger det mere i retning af MAR. MNAR kan ikke afvises sikkert uden domæneviden, fordi den manglende værdi netop ikke observeres.

**Formål:** At visualisere hvor i salgsdatasættet de manglende værdier ligger, og om flere variable mangler samtidig.

**Metode:** Der laves først en missingness matrix, hvor observerede og manglende værdier vises pr. kolonne og observation. Derefter beregnes en korrelationsmatrix for missingness-indikatorerne, så eventuelle sammenhænge mellem mangler i forskellige variable kan vurderes.



In [ ]:
# Missingness matrix: gul = værdi findes, mørk/sort = mangler
missing_matrix = sales_nan.notna().astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(missing_matrix.T, aspect="auto", interpolation="nearest")
ax.set_yticks(range(len(sales_nan.columns)))
ax.set_yticklabels(sales_nan.columns)
ax.set_xlabel("observationer")
ax.set_ylabel("variable")
ax.set_title("Missingness matrix: 1 = observeret (Gul farve), 0 = mangler (Sort farve)")
plt.show()
plt.close("all")

# Korrelation mellem indikatorer for manglende værdier
missing_indicator = sales_nan.isna().astype(int)
missing_corr = missing_indicator.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(missing_corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(missing_corr.columns)))
ax.set_xticklabels(missing_corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(missing_corr.columns)))
ax.set_yticklabels(missing_corr.columns)
ax.set_title("Korrelation mellem missingness-indikatorer")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
plt.close("all")

display(missing_corr.round(3))


**Figurfortolkning:** Missingness matrixen viser, at de manglende værdier er spredt relativt tilfældigt over observationerne, uden tydelige blokke eller systematiske mønstre. Korrelationsmatricen mellem missingness-indikatorerne viser desuden, at der ikke er stærke sammenhænge mellem, hvilke variable der mangler samtidig.

## 1.3 Kvantitativ vurdering af missingness: kan mangler forudsiges?

**Metodeidé:** For hver kolonne med manglende værdier oprettes en binær targetvariabel: `1 = værdien mangler`, `0 = værdien findes`. Derefter trænes en lille classifier på de øvrige kolonner. Hvis modellen kan forudsige missingness markant bedre end tilfældigt, tyder det på, at manglen afhænger af observerede variable og dermed har et MAR-præg.

**Fortolkning af AUC:**

- ca. 0.50: missingness ligner tilfældig variation i forhold til de observerede variable.
- 0.60–0.70: svag/moderat struktur.
- over 0.70: tydelig struktur og sandsynligt MAR.

Dette er ikke et matematisk bevis for MCAR/MAR/MNAR, men et praktisk diagnostisk værktøj.

**Formål:** At undersøge om manglende værdier i hver kolonne kan forudsiges ud fra de øvrige observerede variable.




In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def missingness_auc_table(df, min_missing=20):
    results = []
    for target_col in df.columns:
        y = df[target_col].isna().astype(int)
        if y.sum() < min_missing or y.nunique() < 2:
            continue
        X = df.drop(columns=[target_col]).copy()
        num_cols = X.select_dtypes(include="number").columns.tolist()
        cat_cols = X.select_dtypes(exclude="number").columns.tolist()
        pre = ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", make_ohe())
            ]), cat_cols)
        ])
        model = Pipeline([
            ("pre", pre),
            ("clf", RandomForestClassifier(n_estimators=40, random_state=RANDOM_STATE, min_samples_leaf=10, n_jobs=1))
        ])
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
        )
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob)
        results.append({
            "kolonne": target_col,
            "missing_count": int(y.sum()),
            "missing_pct": round(y.mean() * 100, 2),
            "AUC_missingness_classifier": round(auc, 3),
            "foreløbig_tolkning": "MCAR-lignende" if auc < 0.60 else ("svag/moderat MAR-struktur" if auc < 0.70 else "tydelig MAR-struktur")
        })
    return pd.DataFrame(results).sort_values("AUC_missingness_classifier", ascending=False)

missingness_eval = missingness_auc_table(sales_nan)
display(missingness_eval)

print("Samlet vurdering:")
if len(missingness_eval) > 0 and missingness_eval["AUC_missingness_classifier"].max() >= 0.60:
    print("Mindst én missingness-indikator kan forudsiges bedre end tilfældigt. Det peger på, at datamanglen ikke er ren MCAR, men har et MAR-præg.")
else:
    print("Missingness kan ikke forudsiges markant bedre end tilfældigt. Det peger på MCAR-lignende mangler i dette datasæt.")
print("MNAR kan ikke afvises endeligt uden viden om den dataindsamlingsproces, der har skabt NaN-værdierne.")

## 1.4 Fordelinger før imputering

**Overvejelse:** Imputering kan forvrænge fordelinger. Derfor visualiseres de numeriske variable før imputering. Hvis en metode fx indsætter medianen mange gange, kan det skabe en kunstig top omkring medianen. Det er især relevant ved variable med mange manglende værdier.

**Formål:** At undersøge de numeriske og kategoriske fordelinger, før der udføres imputering.

**Metode:** Numeriske variable vises med histogrammer, og kategoriske variable opsummeres med antal pr. kategori inklusiv manglende værdier. Det giver et udgangspunkt for at vurdere, hvordan imputering senere påvirker datasættets struktur.

**Fortolkning:** Fordelingerne viser, om variable er skæve, har tydelige koncentrationer eller indeholder mange manglende værdier. Det er relevant, fordi simple imputeringer som median/mode kan ændre fordelingen, især hvis en kolonne har mange NaNs.


In [ ]:
num_cols_sales = sales_nan.select_dtypes(include="number").columns.tolist()
cat_cols_sales = sales_nan.select_dtypes(exclude="number").columns.tolist()

fig, axes = plt.subplots(len(num_cols_sales), 1, figsize=(10, 3 * len(num_cols_sales)))
if len(num_cols_sales) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols_sales):
    ax.hist(sales_nan[col].dropna(), bins=40, alpha=0.85)
    ax.set_title(f"Fordeling før imputering: {col}")
    ax.set_xlabel(f"{col}")
    ax.set_ylabel("Antal observationer")
plt.tight_layout()
plt.show()
plt.close("all")

# Kategoriske fordelinger
for col in cat_cols_sales:
    print(f"\n{col}")
    display(sales_nan[col].value_counts(dropna=False).to_frame("count"))

## 1.5 Implementering af imputeringsmetoder

Der implementeres tre metoder:

1. **Simple median/mode**: hurtig og robust baseline.
2. **Random sampling**: bevarer i højere grad kolonnefordelingen, fordi værdier samples fra observerede værdier i samme variabel.
3. **Iterativ/regressionsbaseret imputering for numeriske variable**: estimerer manglende numeriske værdier ud fra de øvrige numeriske variable. Kategoriske variable håndteres med mode.

**Overvejelse:** Hvis datamanglen er MCAR-lignende, kan simple metoder være nok. Hvis den har MAR-præg, kan regressionsbaserede eller iterative metoder være mere relevante, fordi de bruger information fra andre variable.

**Formål:** At implementere de imputeringsmetoder, der senere skal sammenlignes både mod referencefilen og i en classifier-test.

**Metode:** Der oprettes tre komplette versioner af salgsdatasættet: median/mode-imputering, random sampling-imputering og iterativ numerisk imputering kombineret med mode for kategoriske variable.

**Fortolkning:** Outputtet viser, om alle manglende værdier er udfyldt efter hver metode. Det er et nødvendigt kontroltrin, før metoderne kan evalueres og bruges i videre analyse.


In [ ]:
# Hjælpefunktioner til imputering

def simple_median_mode_impute(df):
    out = df.copy()
    num_cols = out.select_dtypes(include="number").columns.tolist()
    cat_cols = out.select_dtypes(exclude="number").columns.tolist()
    for col in num_cols:
        out[col] = out[col].fillna(out[col].median())
    for col in cat_cols:
        out[col] = out[col].fillna(out[col].mode(dropna=True)[0])
    return out


def random_sampling_impute(df, random_state=42):
    local_rng = np.random.default_rng(random_state)
    out = df.copy()
    for col in out.columns:
        mask = out[col].isna()
        observed = out.loc[~mask, col].dropna().to_numpy()
        if mask.sum() > 0 and len(observed) > 0:
            out.loc[mask, col] = local_rng.choice(observed, size=mask.sum(), replace=True)
    return out


def iterative_numeric_mode_categorical_impute(df, max_iter=10):
    out = df.copy()
    num_cols = out.select_dtypes(include="number").columns.tolist()
    cat_cols = out.select_dtypes(exclude="number").columns.tolist()
    if num_cols:
        imp = IterativeImputer(max_iter=max_iter, random_state=RANDOM_STATE, initial_strategy="median")
        out[num_cols] = imp.fit_transform(out[num_cols])
    for col in cat_cols:
        out[col] = out[col].fillna(out[col].mode(dropna=True)[0])
    return out

sales_simple = simple_median_mode_impute(sales_nan)
sales_sampling = random_sampling_impute(sales_nan, random_state=RANDOM_STATE)
sales_iterative = iterative_numeric_mode_categorical_impute(sales_nan, max_iter=10)

print("Manglende værdier efter imputering:")
summary_after = pd.DataFrame({
    "Simple median/mode": sales_simple.isna().sum(),
    "Random sampling": sales_sampling.isna().sum(),
    "Iterativ numeric + mode cat": sales_iterative.isna().sum()
})
display(summary_after)

## 1.6 Sammenligning mod referencefil uden NaNs

**Overvejelse:** Referencefilen gør det muligt at kontrollere imputeringskvaliteten på de celler, der mangler i NaN-versionen. Den bruges kun som evalueringsgrundlag og ikke som input til selve imputeringen.



**Metode:** Kun de tidligere manglende celler evalueres. Numeriske variable vurderes med MAE/RMSE, og kategoriske variable vurderes med accuracy. Resultaterne samles pr. metode.

**Fortolkning:** Lavere MAE/RMSE betyder bedre numerisk imputering, mens højere accuracy betyder bedre kategorisk imputering. Referencefilen bruges kun som evalueringskontrol og ikke som input til selve imputeringen.

**Fortolkning:** Lavere numeriske fejl og højere accuracy peger på en bedre imputering. Resultatet bør dog vurderes sammen med fordelingsplots og classifier-testen.


In [ ]:
# Sørg for samme kolonner og rækkeorden
assert list(sales_nan.columns) == list(sales_full.columns), "Kolonnerne matcher ikke mellem NaN- og referencefil."
assert len(sales_nan) == len(sales_full), "Antal rækker matcher ikke mellem NaN- og referencefil."

imputed_versions = {
    "Simple median/mode": sales_simple,
    "Random sampling": sales_sampling,
    "Iterativ numeric + mode cat": sales_iterative
}

quality_rows = []
for method_name, df_imp in imputed_versions.items():
    for col in sales_nan.columns:
        mask = sales_nan[col].isna()
        if mask.sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(sales_full[col]):
            err = df_imp.loc[mask, col].astype(float) - sales_full.loc[mask, col].astype(float)
            quality_rows.append({
                "metode": method_name,
                "kolonne": col,
                "type": "numerisk",
                "n_eval": int(mask.sum()),
                "MAE": float(np.mean(np.abs(err))),
                "RMSE": float(np.sqrt(np.mean(err ** 2))),
                "Accuracy": np.nan
            })
        else:
            acc = (df_imp.loc[mask, col].astype(str).values == sales_full.loc[mask, col].astype(str).values).mean()
            quality_rows.append({
                "metode": method_name,
                "kolonne": col,
                "type": "kategorisk",
                "n_eval": int(mask.sum()),
                "MAE": np.nan,
                "RMSE": np.nan,
                "Accuracy": float(acc)
            })

imputation_quality = pd.DataFrame(quality_rows)
display(imputation_quality.round(4))

# Samlet overblik pr. metode
method_summary = imputation_quality.groupby("metode").agg(
    mean_MAE=("MAE", "mean"),
    mean_RMSE=("RMSE", "mean"),
    mean_categorical_accuracy=("Accuracy", "mean")
).reset_index()
display(method_summary.round(4))

**Fortolkning af metodevalget:** Den bedste imputeringsmetode vælges ikke kun ud fra én tabel. Numeriske fejlmål viser hvor tæt metoden kommer på referenceværdierne, mens fordelingsplots og classifier-testen viser, om metoden også giver et brugbart datasæt til videre analyse.


## 1.7 Visualisering før og efter imputering

**Overvejelse:** En metode kan give god modelperformance, men stadig forvrænge datastrukturen. Derfor sammenlignes numeriske fordelinger visuelt før og efter imputering. Random sampling forventes ofte at bevare fordelingen bedre end ren medianimputering, mens Iterativ imputering kan bevare relationer mellem numeriske variable.

**Formål:** At kontrollere om imputeringsmetoderne ændrer de numeriske fordelinger på en tydelig måde.

**Metode:** For hver numerisk variabel sammenlignes histogrammet for de observerede værdier før imputering med de tre imputerede datasæt. På den måde kan både fordeling og eventuelle kunstige koncentrationer ses visuelt.

**Fortolkning:** Median/mode kan give ekstra koncentration omkring medianen, mens random sampling ofte bevarer den marginale fordeling bedre. Iterativ imputering kan give mere realistiske numeriske værdier, hvis der er brugbare relationer mellem variable.


In [ ]:
for col in num_cols_sales:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(sales_nan[col].dropna(), bins=40, alpha=0.45, label="Observeret før imputering")
    ax.hist(sales_simple[col], bins=40, alpha=0.35, label="Simple median/mode")
    ax.hist(sales_sampling[col], bins=40, alpha=0.35, label="Random sampling")
    ax.hist(sales_iterative[col], bins=40, alpha=0.35, label="Iterativ numeric + mode cat")
    ax.set_title(f"Fordeling før/efter imputering: {col}")
    ax.set_xlabel(f"{col}")
    ax.set_ylabel("Antal observationer")
    ax.legend()
    plt.show()
plt.close("all")

### Konklusion på fordeling før og efter imputering

Histogrammerne viser, at imputeringsmetoderne påvirker fordelingerne forskelligt.

Median/mode-imputering skaber tydelige toppe omkring én central værdi, især for `Sales_Before`, `Sales_After` og tilfredshedsvariablene. Det skyldes, at mange manglende værdier erstattes med samme medianværdi. Metoden er derfor simpel og brugbar som baseline, men kan give en kunstig fordeling.

Random sampling bevarer den oprindelige fordeling bedre, fordi manglende værdier erstattes med tilfældige observerede værdier fra samme kolonne. Til gengæld bruger metoden ikke sammenhænge mellem variable.

Iterativ imputering giver generelt en mere naturlig fordeling end median/mode, fordi værdierne estimeres ud fra de øvrige variable. Den undgår derfor nogle af de kunstige toppe.

For tilfredshedsvariablene ses mange værdier tæt på 100, hvilket tyder på en skæv fordeling og en øvre grænse. Det bør tages med i fortolkningen.

Samlet vurderes iterativ imputering som den mest hensigtsmæssige metode til de numeriske variable, mens median/mode primært fungerer som en simpel baseline.

## 1.8 Classifier med og uden imputering

**Formål:** Opgaven beder om at sammenligne metoderne fx ved brug af en classifier med/uden imputering. Her bruges `Purchase_Made` som target.

**Sammenligningsdesign:**

- **Uden imputering / complete-case**: rækker med manglende features fjernes.
- **Med imputering**: alle rækker med kendt target kan bruges, fordi manglende features udfyldes.
- **Reference uden NaNs**: den komplette fil bruges som en ekstra øvre kontrolbaseline. Den bør ikke ses som realistisk tilgængelig i en rigtig NaN-opgave.

**Evalueringsmål:** Accuracy, precision, recall og F1. F1 er særligt nyttig, hvis klasserne ikke er helt balancerede.

In [ ]:
def evaluate_classifier_with_pipeline(X_train, X_test, y_train, y_test, numeric_transformer, categorical_transformer, name):
    num_cols = X_train.select_dtypes(include="number").columns.tolist()
    cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
    pre = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ])
    clf = Pipeline([
        ("pre", pre),
        ("model", RandomForestClassifier(n_estimators=60, random_state=RANDOM_STATE, min_samples_leaf=5, n_jobs=1))
    ])
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    return {
        "metode": name,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "accuracy": accuracy_score(y_test, pred),
        "precision_yes": precision_score(y_test, pred, pos_label="Yes", zero_division=0),
        "recall_yes": recall_score(y_test, pred, pos_label="Yes", zero_division=0),
        "f1_yes": f1_score(y_test, pred, pos_label="Yes", zero_division=0),
    }


def ohe_only_transformer():
    return make_ohe()

# Brug kun rækker hvor target findes i NaN-versionen.
target_col = "Purchase_Made"
feature_cols = [c for c in sales_nan.columns if c != target_col]
mask_target = sales_nan[target_col].notna()
X = sales_nan.loc[mask_target, feature_cols]
y = sales_nan.loc[mask_target, target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

results = []

# 1) Uden imputering: complete-case på train/test efter split
train_cc_mask = X_train.notna().all(axis=1)
test_cc_mask = X_test.notna().all(axis=1)
X_train_cc, y_train_cc = X_train.loc[train_cc_mask], y_train.loc[train_cc_mask]
X_test_cc, y_test_cc = X_test.loc[test_cc_mask], y_test.loc[test_cc_mask]

results.append(evaluate_classifier_with_pipeline(
    X_train_cc, X_test_cc, y_train_cc, y_test_cc,
    numeric_transformer="passthrough",
    categorical_transformer=ohe_only_transformer(),
    name="Uden imputering (complete-case)"
))

# 2) Simple median/mode
results.append(evaluate_classifier_with_pipeline(
    X_train, X_test, y_train, y_test,
    numeric_transformer=SimpleImputer(strategy="median"),
    categorical_transformer=Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]),
    name="Simple median/mode"
))

# 3) Iterativ numerisk + mode kategorisk
results.append(evaluate_classifier_with_pipeline(
    X_train, X_test, y_train, y_test,
    numeric_transformer=IterativeImputer(max_iter=10, random_state=RANDOM_STATE, initial_strategy="median"),
    categorical_transformer=Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]),
    name="Iterativ numeric + mode cat"
))

# 4) Random sampling: fit/sampling fra træningsdata, derefter classifier uden imputere i pipeline
X_train_sampling = random_sampling_impute(X_train, random_state=RANDOM_STATE)
# For test samples værdier fra træningsfordelingen for ikke at bruge testfordelingen til imputering
X_test_sampling = X_test.copy()
local_rng = np.random.default_rng(RANDOM_STATE)
for col in X_test_sampling.columns:
    mask = X_test_sampling[col].isna()
    observed_train = X_train[col].dropna().to_numpy()
    if mask.sum() > 0 and len(observed_train) > 0:
        X_test_sampling.loc[mask, col] = local_rng.choice(observed_train, size=mask.sum(), replace=True)

results.append(evaluate_classifier_with_pipeline(
    X_train_sampling, X_test_sampling, y_train, y_test,
    numeric_transformer="passthrough",
    categorical_transformer=ohe_only_transformer(),
    name="Random sampling"
))

# 5) Reference uden NaNs på samme indeks som rækkerne med kendt target i NaN-filen
X_ref = sales_full.loc[mask_target, feature_cols]
y_ref = sales_full.loc[mask_target, target_col]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_ref, y_ref, test_size=0.25, random_state=RANDOM_STATE, stratify=y_ref
)
results.append(evaluate_classifier_with_pipeline(
    Xr_train, Xr_test, yr_train, yr_test,
    numeric_transformer="passthrough",
    categorical_transformer=ohe_only_transformer(),
    name="Referencefil uden NaNs"
))

classifier_results = pd.DataFrame(results).sort_values("f1_yes", ascending=False)
display(classifier_results.round(4))

**Fortolkning af classifier-tabellen:** Tabellen sorteres efter `f1_yes`, så den øverste metode er den bedste i den aktuelle kørsel målt på F1 for klassen `Yes`. Resultatet skal ikke alene afgøre metodevalget, men bruges sammen med imputeringsfejlene fra referencefilen og de visuelle fordelinger. Det er derfor relevant både at vurdere modelperformance og om imputeringen bevarer datastrukturen.


## 1.9 Strukturering i relationel SQLite-database

**Overvejelse:** Salgsdatasættet ligger oprindeligt som én flad CSV-fil. For at demonstrere relationel struktur opdeles data i tabeller, der forbindes med nøgler. Da datasættet ikke har et naturligt kunde-id, oprettes `customer_id` ud fra rækkeindekset.

**Formål:** At vise hvordan salgsdatasættet kan struktureres i en relationel SQLite-database.

**Metode:** Den iterative imputerede version bruges som komplet datagrundlag. Der oprettes tabellerne `customers` og `sales_experiment`, hvor `customer_id` bruges som forbindelse mellem tabellerne. Til sidst udføres en JOIN-forespørgsel som kontrol.

**Fortolkning:** Skema-outputtet og JOIN-resultatet viser, at data kan forbindes og analyseres på tværs af tabeller.

In [ ]:
# Brug den iterative imputationsversion som analyseret/komplet datasæt til databaseeksemplet
sales_db_df = sales_iterative.copy().reset_index().rename(columns={"index": "customer_id"})
sales_db_df["customer_id"] = sales_db_df["customer_id"].astype(int)

# SQLite-database oprettes i samme mappe som notebooken
out_dir = Path.cwd() if Path.cwd().exists() else Path("/mnt/data")
if not (out_dir / "Sales_with_NaNs_v1.3.csv").exists() and Path("/mnt/data").exists():
    out_dir = Path("/mnt/data")

db_path = out_dir / "sales_relational.sqlite"
if db_path.exists():
    db_path.unlink()

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_segment TEXT NOT NULL
);
""")

cur.execute("""
CREATE TABLE sales_experiment (
    record_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    group_label TEXT NOT NULL,
    sales_before REAL,
    sales_after REAL,
    satisfaction_before REAL,
    satisfaction_after REAL,
    purchase_made TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
""")

customers = sales_db_df[["customer_id", "Customer_Segment"]].rename(columns={"Customer_Segment": "customer_segment"})
customers.to_sql("customers", conn, if_exists="append", index=False)

experiment = sales_db_df[[
    "customer_id", "Group", "Sales_Before", "Sales_After",
    "Customer_Satisfaction_Before", "Customer_Satisfaction_After", "Purchase_Made"
]].rename(columns={
    "Group": "group_label",
    "Sales_Before": "sales_before",
    "Sales_After": "sales_after",
    "Customer_Satisfaction_Before": "satisfaction_before",
    "Customer_Satisfaction_After": "satisfaction_after",
    "Purchase_Made": "purchase_made"
})
experiment.to_sql("sales_experiment", conn, if_exists="append", index=False)

conn.commit()

# Kontrol af skemaer
schema_tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
display(schema_tables)

for table in ["customers", "sales_experiment"]:
    print(f"\nSkema for {table}")
    display(pd.read_sql_query(f"PRAGMA table_info({table});", conn))

print("Foreign keys i sales_experiment:")
display(pd.read_sql_query("PRAGMA foreign_key_list(sales_experiment);", conn))

# Eksempel på relationel forespørgsel med JOIN
query = """
SELECT
    c.customer_segment,
    e.group_label,
    COUNT(*) AS n,
    AVG(e.sales_before) AS avg_sales_before,
    AVG(e.sales_after) AS avg_sales_after,
    AVG(e.satisfaction_after) AS avg_satisfaction_after,
    SUM(CASE WHEN e.purchase_made = 'Yes' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS purchase_rate
FROM sales_experiment e
JOIN customers c ON e.customer_id = c.customer_id
GROUP BY c.customer_segment, e.group_label
ORDER BY c.customer_segment, e.group_label;
"""
join_summary = pd.read_sql_query(query, conn)
display(join_summary.round(3))

conn.close()
print(f"Database gemt som: {db_path}")

### Delkonklusion for Del 1

- Manglende værdier findes i både numeriske og kategoriske variable, og andelen varierer mellem kolonner.
- Missingness-vurderingen bør ikke kun baseres på én visualisering. Kombinationen af missingness matrix, korrelationsmatrix og missingness-classifier giver et mere nuanceret grundlag.
- AUC-testen er kun en praktisk indikator og beviser ikke MCAR, MAR eller MNAR.
- AUC-værdierne ligger mellem ca. 0,484 og 0,510. Det betyder, at missingness ikke kan forudsiges bedre end tilfældigt ud fra de observerede variable. Datasættet vurderes derfor som MCAR-lignende, men MNAR kan ikke afvises uden viden om dataindsamlingen.
- Den iterative metode giver lavest numerisk fejl og er derfor bedst til de numeriske variable. For kategoriske variable giver median/mode og iterativ metode samme resultat, fordi begge bruger mode-imputering for kategorier.
- Median/mode er en stærk baseline, men kan forvrænge numeriske fordelinger ved at samle mange observationer omkring medianen.
- Random sampling kan bevare marginalfordelingen bedre, men bruger ikke relationer mellem variable.
- Regressions-/iterativ imputering kan være mere passende ved MAR-lignende mønstre, fordi metoden bruger lighed mellem observationer.
- Classifier-resultatet viser den praktiske effekt: imputering gør det muligt at bruge flere rækker end complete-case og kan derfor forbedre modelgrundlaget, men kvaliteten afhænger af imputeringsmetoden.

# Del 2 – Kortsigtede udsving i tidsseriedata og vinduesmetoden

For tidsserier kan en del af variationen skyldes hurtige dag-til-dag-udsving omkring en langsommere trend. I projektet bruges et glidende gennemsnit som en simpel vinduesmetode til at udglatte klimadata. Metoden skal reducere kortsigtede variationer, men samtidig bevare de langsommere mønstre i data.

Vinduesstørrelsen vælges til 7 dage, fordi datasættet består af daglige målinger. Et uge-vindue giver derfor en naturlig udglatning uden at fjerne den overordnede struktur fuldstændigt.

**Formål:** At undersøge, om klimadata indeholder kortsigtede udsving / støjlignende variation, og om et glidende gennemsnit kan reducere dem uden at fjerne den overordnede struktur.

**Metode:** Train- og testfilerne kombineres, datoer konverteres til `datetime`, data sorteres kronologisk, og dubletdatoen fjernes. Derefter visualiseres de rå tidsserier, outliers i `meanpressure` håndteres særskilt, og effekten evalueres med residualer, daglige ændringer og korrelation.

**Fortolkning:** Del 2 viser, at metoden bevarer de langsommere mønstre, mens de hurtige udsving dæmpes. Det gør vinduesmetoden egnet som praktisk præprocessering i denne opgave.

## 2.1 Indlæsning og klargøring af tidsserier

**Formål:** At samle hele tidsforløbet i én tidsserie, så analysen kan udføres kronologisk på tværs af train og test.

**Metode:** Train- og testfiler kombineres, `date` konverteres til datoformat, datasættet sorteres efter dato, og dubletdatoen 2017-01-01 fjernes.

**Fortolkning:** Den kronologiske struktur bliver korrekt, og der sikres ét sammenhængende tidsforløb, før der laves visualisering og smoothing.

In [ ]:
climate_train = pd.read_csv(climate_train_path)
climate_test = pd.read_csv(climate_test_path)

climate = pd.concat([climate_train, climate_test], ignore_index=True)
climate["date"] = pd.to_datetime(climate["date"])
climate = climate.sort_values("date").reset_index(drop=True)

climate = climate.drop_duplicates(subset="date", keep="first").reset_index(drop=True)

numeric_cols_climate = climate.select_dtypes(include="number").columns.tolist()

print("Tidsperiode:", climate["date"].min().date(), "til", climate["date"].max().date())
print("Antal observationer:", len(climate))
print("Numeriske tidsserier:", numeric_cols_climate)
display(climate.head())
display(climate.describe().T.round(3))

## 2.2 Visualisering af rå tidsserier

**Formål:** At få et første visuelt indtryk af niveauer, variation og mulige afvigelser i de enkelte klimavariable.

**Metode:** De rå tidsserier plottes direkte uden smoothing, så både langsomme trends og hurtige udsving kan ses.

**Fortolkning:** Plottet viser, at serierne har tydelige tidsmønstre, men også kortsigtede udsving / støjlignende variation, som senere vurderes mere systematisk.

In [ ]:
fig, axes = plt.subplots(len(numeric_cols_climate), 1, figsize=(14, 3.2 * len(numeric_cols_climate)), sharex=True)
if len(numeric_cols_climate) == 1:
    axes = [axes]

for ax, col in zip(axes, numeric_cols_climate):
    ax.plot(climate["date"], climate[col], linewidth=1)
    ax.set_title(f"Rå tidsserie: {col}")
    ax.set_xlabel("Dato")
    ax.set_ylabel(f"{col}")
plt.tight_layout()
plt.show()
plt.close("all")

**Figurfortolkning:** De rå tidsserier viser både langsommere sæson- og trendmønstre samt hurtige udsving fra dag til dag. Figuren fungerer derfor som udgangspunkt for den efterfølgende vurdering af støjlignende variation og behovet for udglatning.

## 2.3 Datafejl/outliers før vurdering af kortsigtede udsving

**Overvejelse:** Kortsigtede udsving / støjlignende variation skal adskilles fra åbenlyse datafejl. Ekstreme fejl i `meanpressure` markeres derfor som manglende og erstattes med tidsinterpolation, før vinduesmetoden anvendes.

Det er en separat oprensning og ikke selve vinduesmetoden. Formålet er at sikre, at vurderingen bygger på realistiske tidsserieværdier frem for enkelte fejlregistreringer.


**Formål:** At finde og korrigere ekstreme outliers i `meanpressure`.

**Metode:** Der bruges median og MAD til at beregne en robust z-score. Værdier med meget høj robust z-score sættes til `NaN` og udfyldes derefter med lineær interpolation over tid.

**Fortolkning:** Outputtet viser de værdier, der behandles som datafejl. Efter interpolation kan `meanpressure` indgå mere retvisende i den videre tidsserieanalyse.


In [ ]:
def robust_outlier_mask(series, threshold=8.0):
    med = series.median()
    mad = np.median(np.abs(series - med))
    if mad == 0:
        return pd.Series(False, index=series.index)
    robust_z = 0.6745 * (series - med) / mad
    return robust_z.abs() > threshold

climate_clean = climate.copy()
pressure_mask = robust_outlier_mask(climate_clean["meanpressure"], threshold=8.0)
pressure_outliers = climate_clean.loc[pressure_mask, ["date", "meanpressure"]]

print("Antal stærke outliers i meanpressure:", int(pressure_mask.sum()))
display(climate_clean.loc[pressure_mask, ["date", "meanpressure"]])

climate_clean.loc[pressure_mask, "meanpressure"] = np.nan
climate_clean["meanpressure"] = climate_clean["meanpressure"].interpolate(method="linear", limit_direction="both")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(climate["date"], climate["meanpressure"], label="Rå meanpressure", alpha=0.55)
ax.plot(climate_clean["date"], climate_clean["meanpressure"], label="Efter outlier-håndtering", linewidth=2)
ax.scatter(pressure_outliers["date"], pressure_outliers["meanpressure"], label="Markerede outliers", zorder=3)
ax.set_title("Meanpressure før og efter håndtering af ekstreme outliers")
ax.set_xlabel("Dato")
ax.set_ylabel("meanpressure")
ax.legend()
plt.tight_layout()
plt.show()

## 2.4 Vurdering af kortsigtede udsving / støjlignende variation gennem visualisering og residualer

**Formål:** At vurdere hvor meget de rå tidsserier afviger fra et lokalt trendestimat.

**Metode:** For hver tidsserie beregnes et 7-dages glidende gennemsnit som trendestimat. Residualen defineres som:

$
\text{kortsigtet udsving}_t = x_t - \text{trend}_t
$

Derefter beregnes en simpel ratio:

$
\text{ratio} = \frac{\sigma(\text{kortsigtede udsving})}{\sigma(x)}
$

**Fortolkning:** Ratioen bruges som et praktisk mål for, hvor markante residualerne er i forhold til seriens samlede variation. Den skal fortolkes sammen med visualiseringerne.

In [ ]:
residual_window = 7
udsving_rows = []

fig, axes = plt.subplots(len(numeric_cols_climate), 2, figsize=(16, 3.6 * len(numeric_cols_climate)), sharex=False)
if len(numeric_cols_climate) == 1:
    axes = np.array([axes])

for i, col in enumerate(numeric_cols_climate):
    s = climate_clean[col]
    trend = s.rolling(window=residual_window, center=True, min_periods=1).mean()
    residual = s - trend
    ratio = residual.std() / s.std()
    label = "lav" if ratio < 0.15 else ("moderat" if ratio < 0.30 else "høj")
    udsving_rows.append({
        "tidsserie": col,
        "std_original": s.std(),
        "std_residual_7d": residual.std(),
        "udsving_ratio": ratio,
        "vurderet_udsving_grad": label
    })
    axes[i, 0].plot(climate_clean["date"], s, label="Original", alpha=0.55)
    axes[i, 0].plot(climate_clean["date"], trend, label="7-dages trend", linewidth=2)
    axes[i, 0].set_xlabel("Dato")
    axes[i, 0].set_ylabel(f"{col}")
    axes[i, 0].set_title(f"{col}: original og 7-dages trend")
    axes[i, 0].legend()
    axes[i, 1].plot(climate_clean["date"], residual, linewidth=1)
    axes[i, 1].axhline(0, linewidth=1)
    axes[i, 1].set_xlabel("Dato")
    axes[i, 1].set_ylabel(f"Residual for {col}")
    axes[i, 1].set_title(f"{col}: residual")

plt.tight_layout()
plt.show()
plt.close("all")

udsving_summary = pd.DataFrame(udsving_rows)
display(udsving_summary.round(4))

**Figurfortolkning:** Venstre side af figuren viser de oprindelige tidsserier sammen med et 7-dages trendestimat. Højre side viser residualerne, altså forskellen mellem den oprindelige serie og trendestimatet. Store residualer betyder, at serien har tydelige kortsigtede udsving omkring den langsommere trend.

## 2.5 Præprocessering med vinduesmetoden

**Valg af vindue:** 7 dage. Datasættet har daglige observationer, og vinduet svarer derfor til cirka en uge. Valget bruges som et enkelt og transparent udgangspunkt for smoothing.

**Metode:**

$
\tilde{x}_t = \frac{1}{M}\sum_{i=t-k}^{t+k}x_i
$

hvor \(M\) er antallet af observationer i vinduet. I midten af tidsserien er \(M\) typisk 7, mens \(M\) kan være mindre ved start og slut på grund af `min_periods=1`. I praksis bruges `rolling(window=7, center=True, min_periods=1)`.

**Bemærkning om `center=True`:** Da `center=True` bruges, beregnes gennemsnittet for en dato ud fra både tidligere og senere observationer. Det er acceptabelt her, fordi formålet er efterfølgende analyse og visualisering. Hvis formålet var forecasting, burde man undgå `center=True`, fordi fremtidige observationer ellers ville indgå i beregningen.


**Formål:** At anvende det valgte 7-dages vindue på de oprensede klimadata.

**Metode:** For hver numerisk tidsserie beregnes et 7-dages centreret glidende gennemsnit med `rolling(window=7, center=True, min_periods=1)`. De oprindelige oprensede serier og de udglattede serier plottes sammen.

**Fortolkning:** Plottene viser effekten af præprocesseringen og bruges til at vurdere, om udglatningen er passende for de enkelte variable.

In [ ]:
window = 7
climate_smoothed = climate_clean.copy()
for col in numeric_cols_climate:
    climate_smoothed[col] = climate_clean[col].rolling(window=window, center=True, min_periods=1).mean()

fig, axes = plt.subplots(len(numeric_cols_climate), 1, figsize=(14, 3.4 * len(numeric_cols_climate)), sharex=True)
if len(numeric_cols_climate) == 1:
    axes = [axes]

for ax, col in zip(axes, numeric_cols_climate):
    ax.plot(climate_clean["date"], climate_clean[col], label="Før vinduesmetoden", alpha=0.45)
    ax.plot(climate_smoothed["date"], climate_smoothed[col], label="Efter 7-dages vinduesmetode", linewidth=2)
    ax.set_title(f"Før og efter vinduesmetoden: {col}")
    ax.set_xlabel("Dato")
    ax.set_ylabel(f"{col} / målt værdi")
    ax.legend()
plt.tight_layout()
plt.show()
plt.close("all")


**Figurfortolkning:** Figurerne viser, at det 7-dages glidende gennemsnit udglatter de hurtige dag-til-dag-udsving. Samtidig kan de overordnede mønstre stadig genkendes, hvilket tyder på, at vinduesmetoden reducerer støjlignende variation uden at fjerne hovedstrukturen i tidsserierne.

## 2.6 Evaluering af vinduesmetoden

**Evalueringsidé:** Effekten af vinduesmetoden vurderes ved at sammenligne standardafvigelsen af daglige ændringer før og efter præprocessering:

$$
\Delta x_t = x_t - x_{t-1}
$$

Derudover bruges korrelationen mellem den oprindelige og den udglattede serie som kontrol for, at hovedmønsteret stadig er genkendeligt.

**Formål:** At kontrollere effekten af vinduesmetoden kvantitativt.

**Metode:** Standardafvigelsen af daglige ændringer beregnes før og efter smoothing. Derudover beregnes korrelationen mellem den oprensede originale serie og den udglattede serie.

**Fortolkning:** Et fald i standardafvigelsen viser, at ændringer fra dag til dag er blevet mindre. En høj korrelation tyder på, at udglatningen ikke har ændret seriens hovedmønster væsentligt.


In [ ]:
eval_rows = []
for col in numeric_cols_climate:
    before_diff_std = climate_clean[col].diff().std()
    after_diff_std = climate_smoothed[col].diff().std()
    reduction = 1 - (after_diff_std / before_diff_std)
    corr = climate_clean[col].corr(climate_smoothed[col])
    eval_rows.append({
        "tidsserie": col,
        "std_daglig_ændring_før": before_diff_std,
        "std_daglig_ændring_efter": after_diff_std,
        "reduktion_i_kortsigtet_variation_pct": reduction * 100,
        "korrelation_original_smoothed": corr
    })

smoothing_eval = pd.DataFrame(eval_rows)
display(smoothing_eval.round(4))

### Delkonklusion for Del 2

- De rå klimadata viser tydelige tidsseriemønstre og kortsigtede udsving / støjlignende variation.
- `meanpressure` indeholder enkelte ekstreme outliers, som først håndteres, fordi de ikke repræsenterer almindelige udsving, men sandsynligvis datafejl.
- Kortsigtede udsving vurderes både visuelt og gennem `udsving_ratio`, hvor residualen fra et 7-dages trendestimat bruges som mål for støjlignende variation.
- Vinduesmetoden med 7-dages glidende gennemsnit reducerer de kortsigtede udsving, hvilket ses visuelt og kvantitativt gennem lavere standardafvigelse i daglige ændringer.
- Ulempen er, at glidende gennemsnit også kan udglatte reelle hurtige ændringer. Derfor er vinduesstørrelsen et kompromis mellem udglatning og bevarelse af information.

# Samlet konklusion

Denne samlede notebook besvarer begge opgavesæt:

1. **Datamangel:** Manglende værdier identificeres, visualiseres og vurderes med missingness-diagnostik. MCAR/MAR/MNAR diskuteres, og vurderingen baseres på både visualisering og modelbaseret missingness-test.
2. **Imputering:** Der anvendes flere imputeringsmetoder: median/mode, random sampling og iterativ/regressionsbaseret imputering. Metoderne sammenlignes både mod referencefilen og gennem classifier-performance.
3. **Relationel struktur:** Salgsdatasættet struktureres relationelt i en SQLite-database med relaterede tabeller, primærnøgle og fremmednøgle.
4. **Tidsseriedata:** Klimadata visualiseres, korte udsving / støjlignende variation estimeres med residualer fra et 7-dages trendestimat, og vinduesmetoden anvendes som præprocessering.
5. **Evaluering:** Resultaterne evalueres med både kvantitative mål og kritiske overvejelser, så notebooken ikke kun viser kode, men også begrunder metodevalg og begrænsninger.

## Kilde

I projektet er følgende kilde anvendt som teoretisk baggrund for dataforbehandling, herunder håndtering af manglende værdier og valg af imputeringsmetoder:

García, S., Luengo, J., & Herrera, F. (2015). *Data Preprocessing in Data Mining*. Springer.

##Github Link 

https://github.com/Ziabo1/AI-og-Data-Miniprojekt1
